## 1. Imports & Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, roc_curve, classification_report
)

import os

# Load training data
data_folder = 'C:/Users/xavie/Y3S1/cs421/project/cs421pain/data'
data = np.load(os.path.join(data_folder, "training_batch_with_labels.npz"))
X_raw = data["X"]
y_raw = data["y"]

print(f"Interactions : {X_raw.shape[0]:,}")
print(f"Users        : {len(np.unique(X_raw[:, 0]))}")
print(f"Items        : {len(np.unique(X_raw[:, 1]))}")
print(f"Anomalous    : {np.sum(y_raw[:, 1] == 1)}")
print(f"Normal       : {np.sum(y_raw[:, 1] == 0)}")


Interactions : 177,346
Users        : 1100
Items        : 993
Anomalous    : 100
Normal       : 1000


In [2]:
# load first_batch_with_labels.npz
data_2 = np.load(os.path.join(data_folder, 'first_batch_with_labels.npz'))
X = data_2['X']
y = data_2['y']
print(f"Interactions : {X.shape[0]:,}")
print(f"Users        : {len(np.unique(X[:, 0]))}")
print(f"Items        : {len(np.unique(X[:, 1]))}")
print(f"Anomalous    : {np.sum(y[:, 1] == 1)}")
print(f"Normal       : {np.sum(y[:, 1] == 0)}")

Interactions : 167,493
Users        : 1100
Items        : 989
Anomalous    : 100
Normal       : 1000


In [3]:
# combine the two datasets
print(type(X))
print(type(y))

# combine np arrays
X_raw = np.concatenate((X, X_raw), axis=0)
y_raw = np.concatenate((y, y_raw), axis=0)
print(f"Interactions : {X_raw.shape[0]:,}")
print(f"Users        : {len(np.unique(X_raw[:, 0]))}")
print(f"Items        : {len(np.unique(X_raw[:, 1]))}")
print(f"Anomalous    : {np.sum(y_raw[:, 1] == 1)}")
print(f"Normal       : {np.sum(y_raw[:, 1] == 0)}")



<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
Interactions : 344,839
Users        : 2200
Items        : 996
Anomalous    : 200
Normal       : 2000


In [4]:
# Returns (rows, columns)
print(X_raw.shape) 
print(y.shape)


(344839, 3)
(1100, 2)


In [5]:
RUN_DESC = 'testing item-deviation features + item-entropy + rating distribution shape + gini-coef'

In [6]:
# -- Shared handcrafted-feature definition ---------------------------------
# vvv EDIT HERE to change features — this is the ONLY place you need to touch vvv

def compute_handcrafted_features(df, recon_error=None):
    """
    Per-user feature DataFrame from a [user, item, rating] DataFrame.

    Features implemented from Burke et al. KDD 2006 (the canonical reference
    for supervised shilling detection):
      Generic:  RDMA, WDMA, WDA, DegSim (Pearson), LengthVar
      Model-specific: FMD, FMTD, MeanVar
    Plus extensions: item entropy, popularity bias, gini, deviation stats.
    """
    item_means  = df.groupby("item")["rating"].mean()
    item_counts = df.groupby("item")["rating"].count()
    global_mean = df["rating"].mean()
    global_mean_profile_len = df.groupby("user")["rating"].count().mean()

    df = df.copy()
    df["deviation"]       = df["rating"] - df["item"].map(item_means)
    df["item_popularity"] = df["item"].map(item_counts)
    # Pre-compute per-interaction RDMA term (|dev| * 1/NR_i)
    df["rdma_term"]       = df["deviation"].abs() / df["item_popularity"].clip(lower=1)
    # Pre-compute per-interaction WDMA term (|dev| * NR_i / sum(NR_i))
    # sum(NR_i) per user computed in agg below; approximate with total interactions
    df["wdma_term"]       = df["deviation"].abs() * df["item_popularity"]

    # ── Identify "target" items per user: items rated at max or min ─────────
    # In push/nuke attacks, the target item always receives the extreme rating.
    # We label items where a user gave the global extreme (0 or 5) as
    # potential target items; the remainder are "filler" items.
    rating_max = df["rating"].max()
    rating_min = df["rating"].min()
    df["is_target"] = (df["rating"] == rating_max) | (df["rating"] == rating_min)
    df["is_filler"] = ~df["is_target"]

    agg = df.groupby("user").agg(
        num_ratings       = ("rating",        "count"),
        mean_rating       = ("rating",        "mean"),
        std_rating        = ("rating",        "std"),
        num_items         = ("item",          "nunique"),
        pct_extreme       = ("rating",        lambda r: ((r == 0) | (r == 5)).mean()),
        pct_zero          = ("rating",        lambda r: (r == 0).mean()),
        pct_five          = ("rating",        lambda r: (r == 5).mean()),
        rating_skewness   = ("rating",        lambda r: r.skew()),
        rating_kurtosis   = ("rating",        lambda r: r.kurtosis()),
        median_rating     = ("rating",        "median"),
        item_entropy      = ("item",          lambda x: -(pd.Series(x.value_counts(normalize=True))
                                                          * np.log2(pd.Series(x.value_counts(normalize=True)) + 1e-9)).sum()),
        mean_deviation    = ("deviation",     "mean"),
        std_deviation     = ("deviation",     "std"),
        abs_deviation     = ("deviation",     lambda x: x.abs().mean()),
        gini_items        = ("item",          lambda x: 1 - sum((pd.Series(x).value_counts(normalize=True))**2)),
        pct_low           = ("rating",        lambda r: (r < 3).mean()),

        # ── RDMA: Rating Deviation from Mean Agreement (Burke et al. 2006) ───
        # Sum of |r_ui - mean_i| / NR_i, divided by profile length.
        # High RDMA = user deviates heavily on low-popularity (target) items.
        rdma              = ("rdma_term",     "mean"),

        # ── WDMA: Weighted Deviation from Mean Agreement ─────────────────────
        # Like RDMA but weights by NR_i — captures deviation on POPULAR items.
        # Complements RDMA: RDMA catches rare-item pushes, WDMA catches
        # bandwagon-style attacks on popular filler items.
        wdma              = ("wdma_term",     lambda x: x.sum() / max(x.index.map(
                                 df.set_index(df.index)["item_popularity"]).sum()
                                 if False else 1, 1)),

        # ── WDA: Weighted Degree of Agreement ────────────────────────────────
        # Mean of (rating / global_mean) * item_popularity-weight.
        # Captures users who consistently align with or diverge from the
        # global rating tendency, weighted by item exposure.
        wda               = ("rating",        lambda r: (r / max(global_mean, 1e-9)).mean()),

        # ── LengthVar: Profile Length Variance ───────────────────────────────
        # How much this user's profile length deviates from the population mean.
        # Attackers often have suspiciously short profiles (minimal filler).
        length_var        = ("rating",        lambda r: abs(len(r) - global_mean_profile_len)
                                                        / max(global_mean_profile_len, 1)),

        # ── MeanVar: Mean of target-item rating variances ────────────────────
        # For each profile, identify extreme-rated items (target candidates)
        # and compute the mean squared deviation of ALL items from item means.
        # Attack profiles show elevated variance on their "selected" items.
        mean_var          = ("deviation",     lambda d: (d ** 2).mean()),

        # ── FMD: Filler Mean Difference ───────────────────────────────────────
        # Mean deviation of filler-item ratings from item global means.
        # Attackers choose filler ratings close to item means to blend in,
        # so genuine users show higher FMD variance on filler items.
        fmd               = ("deviation",     lambda d: d[df.loc[d.index, "is_filler"]].mean()
                                                        if df.loc[d.index, "is_filler"].any() else 0),

        # ── FMTD: Filler Mean Target Difference ──────────────────────────────
        # Difference between mean filler rating and mean target (extreme) rating.
        # In push attacks, target items get max rating while fillers get ~mean,
        # producing large FMTD. Genuine users show smaller differences.
        fmtd              = ("rating",        lambda r: (
                                r[df.loc[r.index, "is_target"]].mean() -
                                r[df.loc[r.index, "is_filler"]].mean()
                                if df.loc[r.index, "is_target"].any() and
                                   df.loc[r.index, "is_filler"].any() else 0)),

        # ── Item popularity bias ──────────────────────────────────────────────
        mean_item_pop     = ("item_popularity", "mean"),
        std_item_pop      = ("item_popularity", "std"),
        pct_popular_items = ("item_popularity", lambda x: (x > x.quantile(0.75)).mean()),
    ).fillna(0)

    if recon_error is not None:
        agg["recon_error"] = recon_error.reindex(agg.index).fillna(0)

    return agg

# ^^^ end of editable section ^^^

## 2. Feature Engineering — Per-User Vectors

We pivot the interaction log into a **user × item** matrix where each cell is the rating given (0 if no interaction). This gives each user a 1000-dimensional feature vector capturing their full rating behaviour.

In [7]:
interactions = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels_df    = pd.DataFrame(y_raw, columns=["user", "label"])
print(labels_df["label"].unique() )

[0 1]


In [8]:
# -- Build per-user feature matrix -----------------------------------------

interactions = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels_df    = pd.DataFrame(y_raw, columns=["user", "label"])

user_item = interactions.pivot_table(
    index="user", columns="item", values="rating", aggfunc="mean"
).fillna(0)

labels_df = labels_df.set_index("user").loc[user_item.index].reset_index()
y = labels_df["label"].values

print(f"User-item matrix shape : {user_item.shape}")
print(f"Label array shape      : {y.shape}")
print(f"Class balance          : {np.bincount(y)}  (normal, anomaly)")

# -- SVD reconstruction error -----------------------------------------------
N_RECON_COMPONENTS = 30
_recon_svd = TruncatedSVD(n_components=N_RECON_COMPONENTS, random_state=42)
_ui_vals   = user_item.values
_approx    = _recon_svd.fit_transform(_ui_vals) @ _recon_svd.components_
_residuals = _ui_vals - _approx
recon_error_train = pd.Series(
    np.mean(_residuals ** 2, axis=1), index=user_item.index, name="recon_error"
)
print(f"\nReconstruction error — mean: {recon_error_train.mean():.4f}  "
      f"std: {recon_error_train.std():.4f}")

# -- Handcrafted features ---------------------------------------------------
agg = compute_handcrafted_features(interactions, recon_error=recon_error_train)
agg = agg.loc[user_item.index]
print(f"\nHandcrafted features: {agg.shape[1]} columns")
print(agg.head())

normal_mask = (y == 0)

# -- Cosine similarity to mean normal user ----------------------------------
from sklearn.metrics.pairwise import cosine_similarity
mean_normal_vec = user_item.values[normal_mask].mean(axis=0, keepdims=True)
cos_sim_vals    = cosine_similarity(user_item.values, mean_normal_vec).ravel()
cos_sim_train   = pd.Series(cos_sim_vals, index=user_item.index)
agg["cosine_sim_to_normal"] = cos_sim_train.reindex(agg.index).fillna(0)
print(f"cosine_sim_to_normal — normal: {cos_sim_vals[normal_mask].mean():.4f}  "
      f"anomaly: {cos_sim_vals[~normal_mask].mean():.4f}")

# -- Mahalanobis distance from normal centroid ------------------------------
from sklearn.covariance import LedoitWolf
lw = LedoitWolf().fit(agg.values[normal_mask])
mahal_vals = np.sqrt(lw.mahalanobis(agg.values))
agg["mahal_dist"] = pd.Series(mahal_vals, index=agg.index).reindex(agg.index).fillna(0)
print(f"mahal_dist — normal: {mahal_vals[normal_mask].mean():.4f}  "
      f"anomaly: {mahal_vals[~normal_mask].mean():.4f}")

# -- DegSim (Pearson-based, as defined in Burke et al. 2006) ----------------
# The original DegSim uses Pearson correlation with the k nearest neighbours,
# NOT cosine similarity. Pearson is mean-centred, making it more sensitive to
# rating pattern shape differences that cosine misses.
# We approximate by using np.corrcoef on the user-item matrix.
# Only well-rated users (non-zero rows) contribute signal.
from sklearn.preprocessing import normalize as _norm_fn

# Pearson: subtract each user's mean before computing dot product
_ui_centered = user_item.values - user_item.values.mean(axis=1, keepdims=True)
_ui_pearson  = _norm_fn(_ui_centered, norm="l2")   # unit-normalise centered rows
_sim_pearson = _ui_pearson @ _ui_pearson.T          # (n_users, n_users) Pearson approx
np.fill_diagonal(_sim_pearson, 0)
_DEGSIM_K    = 20   # Burke et al. use k=20
degsim_vals  = np.sort(_sim_pearson, axis=1)[:, -_DEGSIM_K:].mean(axis=1)
agg["degsim_pearson"] = pd.Series(degsim_vals, index=user_item.index).reindex(agg.index).fillna(0)
print(f"degsim_pearson — normal: {degsim_vals[normal_mask].mean():.4f}  "
      f"anomaly: {degsim_vals[~normal_mask].mean():.4f}")
print("(Per literature: ATTACKERS should have LOW DegSim — filler ratings are")
print(" randomly chosen so their profiles are dissimilar to genuine neighbours)")

# -- Co-rating overlap (structural graph proxy) -----------------------------
_ui_binary  = (user_item.values > 0).astype(np.float32)
_ui_sp_norm = _norm_fn(_ui_binary, norm="l2")
_overlap    = _ui_sp_norm @ _ui_sp_norm.T
np.fill_diagonal(_overlap, 0)
_top5_ol    = np.sort(_overlap, axis=1)[:, -5:].mean(axis=1)
agg["top5_corating_overlap"] = pd.Series(_top5_ol, index=user_item.index).reindex(agg.index).fillna(0)
print(f"top5_corating_overlap — normal: {_top5_ol[normal_mask].mean():.4f}  "
      f"anomaly: {_top5_ol[~normal_mask].mean():.4f}")

print(f"\nTotal handcrafted features: {agg.shape[1]}")

User-item matrix shape : (2200, 996)
Label array shape      : (2200,)
Class balance          : [2000  200]  (normal, anomaly)

Reconstruction error — mean: 0.7597  std: 0.3916

Handcrafted features: 27 columns
      num_ratings  mean_rating  std_rating  num_items  pct_extreme  pct_zero  \
user                                                                           
100            88     3.147727    1.169928         88     0.113636  0.045455   
101           231     3.406926    0.854016        231     0.095238  0.008658   
102            98     3.673469    1.072385         98     0.193878  0.000000   
103           103     3.000000    1.290994        103     0.126214  0.058252   
104            89     3.719101    0.988442         89     0.235955  0.000000   

      pct_five  rating_skewness  rating_kurtosis  median_rating  ...     wdma  \
user                                                             ...            
100   0.068182        -0.955265         0.852420            3.0  ..

In [14]:
# convert to csv
agg_with_labels = agg.copy()
agg_with_labels["class"] = y   # 0 = normal, 1 = anomaly

# NORMALIZE — this is what was missing
feature_cols = [c for c in agg_with_labels.columns if c != "class"]
agg_with_labels[feature_cols] = StandardScaler().fit_transform(
    agg_with_labels[feature_cols]
)

agg_with_labels.to_csv("dataset/user_features_with_labels.csv", index=False) # remove index